# TTS wolof — évaluation comparative de modèles (S2-J4)

## Contexte

Premier chantier TTS sur le wolof, sur le modèle de la comparaison ASR (S2-J3). Le plan initial
reposait sur `facebook/mms-tts-wol` : hypothèse invalidée, MMS-TTS ne couvre pas le wolof.

| Modèle | Architecture | Licence | Voix |
|---|---|---|---|
| `AIHubSN/Kiriku-Wolof-TTS` | VITS mono-locuteur (Coqui) | Apache 2.0 | figée |
| `soynade-research/Oolel-Voices` | architecture propre (~0.5B) | AGPL-3.0 | clonage vocal |
| `bilalfaye/speecht5_tts-wolof-v0.2` | SpeechT5 + HiFi-GAN | MIT | speaker embedding |
| `galsenai/xTTS-v2-wolof` | Coqui xTTS v2 | non commercial (CPML) | clonage vocal — *à tester* |


Ce notebook ne construit pas le corpus, il **charge** l'artefact figé produit en amont
(`tts/corpus_test_tts_wolof.json`, 36 phrases, 5 catégories) : tous les modèles sont testés sur
exactement le même input.

## Objectif

- Synthétiser le corpus sur chaque modèle, **un seul modèle en VRAM à la fois**
  (charge → synthèse → écriture disque → décharge)
- Noter intelligibilité et naturel par catégorie (MOS simplifié, évaluateur unique — limite
  assumée, cohérente avec S1-J3 et S2-J3)
- Arrêter un modèle en croisant performance mesurée et licence
- Alimenter la comparaison FR/wolof du rapport baseline S4

## Environnement — contournements à conserver

- **`coqui-tts`** et non `TTS` : le paquet Coqui original s'arrête à Python 3.11
- **Shim `isin_mps_friendly`** : retiré en `transformers` v5, importé par le chemin xTTS de
  `coqui-tts`. `torch.isin` est l'équivalent
- **Snapshot Oolel sur `sys.path`** : son code distant suppose un layout de cache antérieur

## Paramètres d'expérience — à déclarer dans la grille

- **Sample rates hétérogènes** : 22 050 Hz (Kiriku, Oolel) vs **16 kHz** (SpeechT5), qui le
  désavantage sur le naturel
- **Voix de référence commune** : le WAV de démo Oolel sert de prompt à Oolel et de source du
  x-vector à SpeechT5. Kiriku a sa voix figée
- **`.lower()` propre à Kiriku** : son vocabulaire n'a pas de majuscules, elles sont supprimées
  silencieusement — contrainte d'intégration réelle en production
- **Embedding substitut (SpeechT5)** : le dépôt n'en fournit pas, la source recommandée
  (CMU ARCTIC) n'est plus chargeable. x-vector SpeechBrain non normalisé
- **`temperature` ignorée par Oolel** : réglages par défaut du modèle
- **Biais des catégories** : `codeswitch` orientée CS technique via liste *ad hoc* (le CS réel
  de KALLAAMA est surtout discursif) ; `lexique_domaine` = questions, pas les réponses que le
  TTS vocalisera en production ; `nombres` curée à la main (bruit tél./fréquences radio)

## Section 0 — Setup

In [ ]:
# Installs — puis RESTART le runtime avant la suite
!pip install -q -U transformers
!pip install -q coqui-tts speechbrain
!pip install -q "librosa>=0.10.2" conformer==0.3.2 torchcodec s3tokenizer num2words
print("→ RESTART le runtime (Exécution > Redémarrer la session), puis reprendre au Setup")

In [ ]:
# transformers v5 a retiré isin_mps_friendly, que coqui-tts (chemin xTTS) importe au chargement.
# torch.isin est l'équivalent — leur propre code le note en TODO.
import torch, transformers.pytorch_utils as pu
if not hasattr(pu, "isin_mps_friendly"):
    pu.isin_mps_friendly = lambda e, t: torch.isin(e, t)

from TTS.utils.synthesizer import Synthesizer
print("import OK")

In [ ]:
from pathlib import Path
import sys, json

# 1. Code + corpus figé : arrivent par git (le JSON est versionné sous tts/)
PROJECT_ROOT = Path("/content/noo-far-pipeline")
if not PROJECT_ROOT.exists():
    !git clone https://github.com/noofar-ia/noo-far-pipeline.git {PROJECT_ROOT}
else:
    !cd {PROJECT_ROOT} && git pull
sys.path.insert(0, str(PROJECT_ROOT))

CORPUS_JSON = PROJECT_ROOT / "tts" / "corpus_test_tts_wolof.json"

# 2. Drive : uniquement pour la SORTIE (les WAV + grilles)
from google.colab import drive
drive.mount('/content/drive')
EVAL_DIR = Path("/content/drive/MyDrive/noo-far-pipeline/tts_eval")
EVAL_DIR.mkdir(parents=True, exist_ok=True)

# 3. Token HF (Kiriku est un dépôt gated : conditions à accepter sur la page HF)
from google.colab import userdata
from huggingface_hub import login

login(token=userdata.get("HF_TOKEN"))

print("corpus figé :", CORPUS_JSON.exists())
print("sortie      :", EVAL_DIR)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Charger le corpus et preparer la sortie pour chaque modèle

with open(CORPUS_JSON, encoding="utf-8") as f:
    corpus_meta = json.load(f)
corpus = {cat: d["phrases"] for cat, d in corpus_meta.items()}

for cat, ph in corpus.items():
    print(f"{cat:20s} : {len(ph):2d}")
print("TOTAL :", sum(len(p) for p in corpus.values()))

import csv

def preparer_sortie(nom_modele):
    """Crée le dossier du modèle et renvoie (dossier, chemin_csv, chemin_manifest)."""
    d = EVAL_DIR / nom_modele
    d.mkdir(parents=True, exist_ok=True)
    return d, EVAL_DIR / f"{nom_modele}_grille.csv", EVAL_DIR / f"{nom_modele}_manifest.json"

def ecrire_grille(chemin_csv, manifest):
    """Grille MOS à remplir en écoutant : 3 dernières colonnes vides."""
    with open(chemin_csv, "w", encoding="utf-8", newline="") as f:
        w = csv.writer(f)
        w.writerow(["id", "categorie", "texte", "wav", "intelligibilite", "naturel", "remarques"])
        for it in manifest:
            w.writerow([it["id"], it["categorie"], it["texte"], it.get("wav", ""), "", "", ""])
    print("grille :", chemin_csv)

## Section 1 — Kiriku-Wolof-TTS (AIHubSN)

In [ ]:
from huggingface_hub import snapshot_download
from TTS.utils.synthesizer import Synthesizer
from IPython.display import Audio, display

ckpt = snapshot_download(repo_id="AIHubSN/Kiriku-Wolof-TTS", local_dir="/content/kiriku")
!ls -lh /content/kiriku

synth = Synthesizer(
    tts_checkpoint="/content/kiriku/model.pth",
    tts_config_path="/content/kiriku/config.json",
    use_cuda=True,
)

# .lower() identique à la passe : le vocab Kiriku n'a pas de majuscules, elles sont
# supprimées silencieusement. Sans ça, le test ne préfigure pas ce que la passe produit.
phrase = corpus["lexique_domaine"][0]
print("\nphrase :", phrase)
wav = synth.tts(phrase.lower())
print("type :", type(wav), "| len :", len(wav), "| sr :", synth.output_sample_rate)
display(Audio(wav, rate=synth.output_sample_rate))

In [ ]:
import numpy as np, soundfile as sf, json, gc, torch

NOM = "kiriku-wolof-vits"
dossier, chemin_csv, chemin_manifest = preparer_sortie(NOM)

manifest = []
for categorie, phrases in corpus.items():
    for i, phrase in enumerate(phrases):
        ident = f"{categorie}_{i:02d}"
        texte_in = phrase.lower()      # vocab Kiriku sans majuscules (cf. carte : lower-casing à l'inférence)
        try:
            wav = np.asarray(synth.tts(texte_in), dtype=np.float32)
            chemin = dossier / f"{ident}.wav"
            sf.write(chemin, wav, synth.output_sample_rate)
            manifest.append({"id": ident, "categorie": categorie, "texte": phrase,
                             "texte_envoye": texte_in, "wav": str(chemin),
                             "sr": synth.output_sample_rate})
        except Exception as e:
            print(f"  ÉCHEC {ident} : {type(e).__name__} — {e}")
            manifest.append({"id": ident, "categorie": categorie, "texte": phrase,
                             "texte_envoye": texte_in, "wav": "", "erreur": str(e)})
    print(f"{categorie:20s} ok")

chemin_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
ecrire_grille(chemin_csv, manifest)

ok = sum(1 for m in manifest if m.get("wav"))
print(f"\n{ok}/36 clips écrits dans {dossier}")
print("prétraitement : .lower() (spécifique Kiriku)")

In [ ]:
del synth
gc.collect()
torch.cuda.empty_cache()
print("VRAM allouée :", torch.cuda.memory_allocated() // 1024**2, "Mo")

## Section 2 — Oolel-Voices (soynade-research)

In [ ]:
VOIX_REF = "/content/oolel_demo_ref.wav"
!wget -q -O {VOIX_REF} "https://huggingface.co/spaces/soynade-research/Oolel-Voices-Demo/resolve/main/8_1_c.wav"
!ls -lh {VOIX_REF}

import sys, torch
from huggingface_hub import snapshot_download
from transformers import AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
path = snapshot_download(repo_id="soynade-research/Oolel-Voices")
if path not in sys.path:
    sys.path.insert(0, path)          # contourne le cache dynamique éclaté par hash

model_oolel = AutoModel.from_pretrained(path, trust_remote_code=True).to(device)
print("chargé")

In [ ]:
import numpy as np, soundfile as sf, json, gc, torch

NOM = "oolel-voices"
dossier, chemin_csv, chemin_manifest = preparer_sortie(NOM)

manifest = []
for categorie, phrases in corpus.items():
    for i, phrase in enumerate(phrases):
        ident = f"{categorie}_{i:02d}"
        try:
            out = model_oolel.generate(phrase, audio_prompt_path=VOIX_REF,
                                       cfg_weight=0.5, exaggeration=0.2, temperature=0.3)
            wav = out.detach().cpu().numpy().squeeze().astype(np.float32)
            chemin = dossier / f"{ident}.wav"
            sf.write(chemin, wav, model_oolel.sr)
            manifest.append({"id": ident, "categorie": categorie, "texte": phrase,
                             "wav": str(chemin), "sr": model_oolel.sr})
        except Exception as e:
            print(f"  ÉCHEC {ident} : {type(e).__name__} — {e}")
            manifest.append({"id": ident, "categorie": categorie, "texte": phrase,
                             "wav": "", "erreur": str(e)})
    print(f"{categorie:20s} ok")

chemin_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
ecrire_grille(chemin_csv, manifest)

ok = sum(1 for m in manifest if m.get("wav"))
print(f"\n{ok}/36 clips écrits dans {dossier}")
print("voix de référence : WAV de démo Oolel | temperature ignorée (défaut modèle)")

In [ ]:
del model_oolel
gc.collect(); torch.cuda.empty_cache()
print("VRAM :", torch.cuda.memory_allocated() // 1024**2, "Mo")
!df -h /content | tail -1

## Section 3 — speecht5_tts-wolof-v0.2 (bilalfaye)

In [ ]:
# Le dépôt SpeechT5 ne fournit pas d'embedding, et la source recommandée (CMU ARCTIC)
# n'est plus chargeable (datasets a supprimé les scripts). Substitut : x-vector SpeechBrain
# dérivé du WAV de référence Oolel → voix commune aux deux modèles.
# Non normalisé : les variantes normalisée et extrait-court ne sonnaient pas mieux.
import torchaudio
from speechbrain.inference import EncoderClassifier

enc = EncoderClassifier.from_hparams("speechbrain/spkrec-xvect-voxceleb",
                                     savedir="/content/spkrec", run_opts={"device": device})
sig, sr = torchaudio.load(VOIX_REF)
if sig.shape[0] > 1:
    sig = sig.mean(0, keepdim=True)
if sr != 16000:
    sig = torchaudio.functional.resample(sig, sr, 16000)

with torch.no_grad():
    spk_brut = enc.encode_batch(sig.to(device)).squeeze(0)
print("speaker embedding :", tuple(spk_brut.shape))

In [ ]:
from transformers import SpeechT5ForTextToSpeech, SpeechT5Processor, SpeechT5HifiGan
from IPython.display import Audio, display

repo    = "bilalfaye/speecht5_tts-wolof-v0.2"
proc    = SpeechT5Processor.from_pretrained(repo)
model   = SpeechT5ForTextToSpeech.from_pretrained(repo).to(device)
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan").to(device)

phrase = corpus["lexique_domaine"][0]
inputs = proc(text=phrase, return_tensors="pt", padding=True,
              truncation=True, max_length=model.config.max_text_positions)
inputs = {k: v.to(device) for k, v in inputs.items()}
with torch.no_grad():
    wav = model.generate(inputs["input_ids"], speaker_embeddings=spk_brut, vocoder=vocoder,
                         num_beams=7, no_repeat_ngram_size=3)
print("type :", type(wav), "| shape :", tuple(wav.shape))
display(Audio(wav.cpu().numpy().squeeze(), rate=16000))

In [ ]:
import numpy as np, soundfile as sf, json, gc, torch

NOM = "speecht5-wolof-v0.2"
dossier, chemin_csv, chemin_manifest = preparer_sortie(NOM)

manifest = []
for categorie, phrases in corpus.items():
    for i, phrase in enumerate(phrases):
        ident = f"{categorie}_{i:02d}"
        try:
            inputs = proc(text=phrase, return_tensors="pt", padding=True,
                          truncation=True, max_length=model.config.max_text_positions)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                out = model.generate(inputs["input_ids"], speaker_embeddings=spk_brut,
                                     vocoder=vocoder, num_beams=7, no_repeat_ngram_size=3)
            wav = out.detach().cpu().numpy().squeeze().astype(np.float32)
            chemin = dossier / f"{ident}.wav"
            sf.write(chemin, wav, 16000)
            manifest.append({"id": ident, "categorie": categorie, "texte": phrase,
                             "wav": str(chemin), "sr": 16000})
        except Exception as e:
            print(f"  ÉCHEC {ident} : {type(e).__name__} — {e}")
            manifest.append({"id": ident, "categorie": categorie, "texte": phrase,
                             "wav": "", "erreur": str(e)})
    print(f"{categorie:20s} ok")

chemin_manifest.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
ecrire_grille(chemin_csv, manifest)

ok = sum(1 for m in manifest if m.get("wav"))
print(f"\n{ok}/36 clips écrits dans {dossier}")
print("embedding : x-vector SpeechBrain du WAV réf. Oolel, NON normalisé | sr 16 kHz")

In [ ]:
del model, vocoder, proc, enc
gc.collect(); torch.cuda.empty_cache()
print("VRAM :", torch.cuda.memory_allocated() // 1024**2, "Mo")

## Section 4 — xTTS-v2-wolof (galsenai) — à tester

Reporté à la semaine prochaine. Galsen IA est la communauté IA de référence au Sénégal, ce qui
donne à ce modèle un intérêt d'écosystème au-delà de sa performance.

**Points à traiter avant de lancer** :
- Licence non commerciale (CPML héritée de Coqui xTTS v2) → non déployable en production
  Ñoo Far quel que soit son score ; mesuré comme borne haute de ce qui est atteignable
- ~7 Go, chargement non standard (gdown selon le README) → vérifier l'espace disque
- Base Coqui, comme Kiriku : tourne dans le même environnement `coqui-tts` + shim
- Clonage vocal → réutiliser `VOIX_REF` (WAV de démo Oolel) pour rester à voix constante

## Section 5 — Grille MOS

Les notes se remplissent dans les CSV par modèle écrits sur Drive
(`tts_eval/<modele>_grille.csv`, une ligne par clip, colonnes `intelligibilite`,
`naturel`, `remarques` vides). Ouvrir dans Sheets, écouter, noter.

### Échelle

**1** incompréhensible · **2** difficile · **3** compréhensible avec effort ·
**4** clair · **5** parfaitement clair et naturel

Deux axes **distincts** : un modèle peut être parfaitement intelligible avec une prosodie
robotique (5/2), ou agréable mais imprécis (3/4).

### Protocole

- Noter **modèle par modèle**, dans l'ordre des catégories — pas en alternant les modèles :
  108 clips en une session, l'oreille se fatigue et devient moins discriminante.
- Écouter chaque clip une fois, noter, passer. Ne pas revenir en arrière pour « comparer » :
  la comparaison se fait à la lecture des moyennes, pas à l'écoute.
- Remplir `remarques` seulement quand quelque chose se produit (nombre épelé, mot avalé,
  artefact) — c'est cette colonne qui fera le rapport, plus que les chiffres.

### À écouter spécifiquement

| Catégorie | Question |
|---|---|
| `lexique_domaine` | le vocabulaire métier (feebar, veterineer, nàmp) est-il prononçable ? |
| `phrases_kallaama` | les disfluences de l'oral spontané passent-elles ? |
| `phrases_nombres` | nombres **vocalisés ou épelés** ? chiffres arabes et mots wolof traités pareil ? |
| `phrases_codeswitch` | les mots français enclavés : prononcés à la française, wolofisés, massacrés ? |
| `wolof_ecrit` | les noms propres et la syntaxe longue tiennent-ils ? |

### Synthèse

Comparer **catégorie par catégorie**, jamais en moyenne globale : les catégories ont des
tailles différentes (10/5/8/8/5) et une moyenne serait tirée par `lexique_domaine`.

| Modèle | Catégorie | Intelligibilité | Naturel | Remarques |
|---|---|---|---|---|

| kiriku-wolof-vits | lexique_domaine |3.5|4 | certain mots français mal|
| kiriku-wolof-vits | phrases_kallaama |3.75 | 4| |
| kiriku-wolof-vits | phrases_nombres |2 |2 |seul les lettres sont bien prononcés |
| kiriku-wolof-vits | phrases_codeswitch |4|4 |le rhytme manque de pauses et de naturel |
| kiriku-wolof-vits | wolof_ecrit |3.5|3.5 | |
| oolel-voices | lexique_domaine |4.5 |4.5 | |
| oolel-voices | phrases_kallaama |3.5|4| |
| oolel-voices | phrases_nombres |2.5|2.5 | |
| oolel-voices | phrases_codeswitch |4.75|4.5 | |
| oolel-voices | wolof_ecrit |4.5 |4.5| |
| speecht5-wolof-v0.2 | lexique_domaine | | | |
| speecht5-wolof-v0.2 | phrases_kallaama | | | |
| speecht5-wolof-v0.2 | phrases_nombres | | | |
| speecht5-wolof-v0.2 | phrases_codeswitch | | | |  
| speecht5-wolof-v0.2 | wolof_ecrit | | | |

**Décision d'architecture** : **Kiriku en socle** pour le pipeline S2-J5/J6, malgré la
meilleure note d'Oolel. Le motif est la **latence** : VITS mono-locuteur non autorégressif
contre 0,5 B autorégressif avec clonage, dans une chaîne où Whisper large et le LLM
s'additionnent déjà, pour un usage où l'éleveur attend au téléphone sans retour visuel.
C'est le seul argument qui porte ce choix — la licence n'en est pas un (le copyleft AGPL est
cohérent avec la gouvernance Ostrom du projet, dont le pipeline a vocation à être ouvert), et
la finetunabilité non plus (le levier TTS est le frontend texte, pas l'adaptation au domaine).

**Oolel gardé en sonde ponctuelle**, substitué par l'adaptateur TTS une fois la chaîne
instrumentée en J6. Si l'écart de latence bout en bout est acceptable, **Oolel redevient le
choix par défaut** : meilleure qualité mesurée, clonage vocal, licence conforme. La décision
ci-dessus est donc provisoire et repose sur une hypothèse non encore vérifiée.

**SpeechT5 éliminé** avant la grille : voix robotique avec écho, trois variantes de speaker
embedding testées sans amélioration. Le défaut est le modèle, pas l'embedding. Une licence
permissive ne sauve pas un modèle inutilisable.

**Faiblesses récurrentes** : [à remplir après la grille — regarder si `nombres` et
`codeswitch` cassent chez les trois (alors c'est un problème de frontend texte, à traiter
dans le code appelant) ou seulement chez l'un (alors c'est le modèle). Distinction
déterminante : le frontend se corrige à coût quasi nul, le modèle non.]

**Contraintes d'intégration** :

- **Kiriku** — `.lower()` obligatoire en amont. Son vocabulaire ne contient aucune majuscule :
  elles sont supprimées **silencieusement**, sans erreur levée, mutilant les noms propres
  (ISRA, toponymes, noms de maladies) que contiendront les réponses RAG. À placer dans
  l'adaptateur TTS, jamais chez l'appelant. Sortie 22 050 Hz.
- **Oolel** — voix de référence (WAV) à embarquer et à figer : le même prompt pour tous les
  appels, sinon la voix varie d'une réponse à l'autre. `trust_remote_code=True` exécute du
  code tiers ; le snapshot doit être ajouté à `sys.path` (leur code suppose un layout de cache
  antérieur). Sortie 22 050 Hz.
- **Commun aux deux** — frontend texte à écrire en amont de l'adaptateur : verbalisation des
  nombres (un TTS entraîné sur graphèmes ne lit pas « 63% »), table de prononciation pour le
  lexique métier, gestion des emprunts français. Agnostique au modèle, donc à écrire une fois.
- **Environnement** — `coqui-tts` (le paquet Coqui original s'arrête à Python 3.11) et shim
  `isin_mps_friendly` (retiré en `transformers` v5, importé par le chemin xTTS de `coqui-tts`).